# V5 Complete Pipeline: Deepfake Detection X.AI


---

## 목차

### Part A: V2 Zero-Shot CLIP
1. 환경 설정 (패키지 설치, 라이브러리)
2. 경로 및 설정
3. 대회 데이터 확인
4. Zero-Shot CLIP 추론 → `sub_V2_clip_L14_v3.csv`

### Part B: 공통 준비
5. 학습 데이터 다운로드 (5개 소스)
6. Face Detection 모델 준비 (MediaPipe)

### Part C: V3 Face-Centric Fine-Tuning
7. V3 데이터 수집 + Face Filter + Distribution Matching
8. V3 모델 학습 (CLIP ViT-L-14, 224px)
9. V3 테스트 추론 (TTA) → `V3_face_centric_tta.csv`

### Part D: V5 Main Pipeline (336px)
10. V5 데이터 수집 + Face Filter + Pre-crop
11. V5 클래스 밸런싱 + Augmentation + DataLoader
12. V5 모델 학습 (CLIP ViT-L-14-336, 336px)
13. V5 테스트 추론 (Face Crop + 5-view TTA) → `V5_final_tta.csv`

### Part E: 앙상블
14. 3-way Rank Ensemble → `V5_ENS_zs_dom_3way.csv`
15. 최종 검증

In [ ]:
# ============================================================
# 1. 필수 패키지 설치
# ============================================================
!pip install kagglehub open_clip_torch mediapipe opencv-python-headless scikit-learn tqdm pandas -q

In [ ]:
# GLES dependency for MediaPipe (Linux)
!apt-get update -qq && apt-get install -y -qq libgles2 libgl1 libegl1 2>/dev/null || true

In [ ]:
# ============================================================
# 2. 라이브러리 / 경로 / 설정
# ============================================================
import os, gc, random, time, json, io, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms

import cv2
import open_clip
import mediapipe as mp
from mediapipe.tasks.python import vision as mp_vision

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---- 재현성 ----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---- 경로 ----
PROJECT   = Path("/workspace/project")
COMP_DATA = Path("/workspace/deepfake-detection-challenge-deep-preview-x-ai-7th")
TEST_DIR  = COMP_DATA / "test"
SUB_DIR   = PROJECT / "submissions"
CKPT_DIR  = PROJECT / "checkpoints"
CACHE_DIR = PROJECT / "cache" / "v5"

for d in [SUB_DIR, CKPT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

# ---- V3 하이퍼파라미터 ----
V3_IMG_SIZE    = 224
V3_BATCH_SIZE  = 32
V3_EPOCHS      = 8
V3_BACKBONE_LR = 2e-6
V3_HEAD_LR     = 1e-3
V3_WEIGHT_DECAY = 0.01
V3_WARMUP_EPOCHS = 1

# ---- V5 하이퍼파라미터 ----
V5_IMG_SIZE    = 336
V5_BATCH_SIZE  = 24
V5_EPOCHS      = 12
V5_PATIENCE    = 4
V5_BACKBONE_LR = 1e-6
V5_HEAD_LR     = 5e-4
V5_WEIGHT_DECAY = 0.02

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# 3. 대회 데이터 확인
# ============================================================
assert COMP_DATA.exists(), f"대회 데이터가 없습니다: {COMP_DATA}"
assert TEST_DIR.exists(), f"test 폴더가 없습니다: {TEST_DIR}"

test_csv = pd.read_csv(COMP_DATA / "test.csv")
sample_sub = pd.read_csv(COMP_DATA / "sample_submission.csv")

test_files = sorted(TEST_DIR.glob("*.png"))
test_ids = [f.stem for f in test_files]

print(f"테스트 이미지 수: {len(test_csv)}")
print(f"테스트 폴더 파일 수: {len(test_files)}")
print(test_csv.head())

---

## Part A: V2 Zero-Shot CLIP → `sub_V2_clip_L14_v3.csv`

CLIP ViT-L-14 (OpenAI pretrained)로 **학습 없이** zero-shot 분류.
간결한 프롬프트 2쌍으로 real/fake 유사도를 계산합니다.

| 프롬프트 | 역할 |
|---------|------|
| "a real face in a photograph" | Real |
| "a photo taken with a real camera of a real person" | Real |
| "a fake face that was generated by a computer" | Fake |
| "a synthetically generated face that is not real" | Fake |

In [ ]:
# ============================================================
# 4. Zero-Shot CLIP 추론 → sub_V2_clip_L14_v3.csv
# ============================================================
print("=" * 80)
print("Part A: V2 Zero-Shot CLIP Inference")
print("=" * 80)

zs_model, _, zs_preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
zs_model = zs_model.to(DEVICE).eval()
zs_tokenizer = open_clip.get_tokenizer("ViT-L-14")

prompts_real = [
    "a real face in a photograph",
    "a photo taken with a real camera of a real person",
]
prompts_fake = [
    "a fake face that was generated by a computer",
    "a synthetically generated face that is not real",
]

with torch.no_grad():
    text_real = zs_tokenizer(prompts_real).to(DEVICE)
    text_fake = zs_tokenizer(prompts_fake).to(DEVICE)
    feat_real = zs_model.encode_text(text_real)
    feat_fake = zs_model.encode_text(text_fake)
    feat_real = F.normalize(feat_real, dim=-1).mean(dim=0, keepdim=True)
    feat_fake = F.normalize(feat_fake, dim=-1).mean(dim=0, keepdim=True)
    feat_real = F.normalize(feat_real, dim=-1)
    feat_fake = F.normalize(feat_fake, dim=-1)

zs_scores = []
ZS_BATCH = 16

for i in tqdm(range(0, len(test_files), ZS_BATCH), desc="ZS inference"):
    batch_files = test_files[i:i+ZS_BATCH]
    images = [zs_preprocess(Image.open(f).convert("RGB")) for f in batch_files]
    batch = torch.stack(images).to(DEVICE)
    with torch.no_grad():
        img_feat = zs_model.encode_image(batch)
        img_feat = F.normalize(img_feat, dim=-1)
        sim_real = (img_feat @ feat_real.T).squeeze(-1)
        sim_fake = (img_feat @ feat_fake.T).squeeze(-1)
        prob_fake = torch.softmax(torch.stack([sim_real, sim_fake], dim=-1) * 100.0, dim=-1)[:, 1]
        zs_scores.extend(prob_fake.cpu().numpy().tolist())

zs_scores = np.array(zs_scores)

# 저장
zs_df = pd.DataFrame({"id": test_ids, "score": zs_scores})
zs_path = SUB_DIR / "sub_V2_clip_L14_v3.csv"
zs_df.to_csv(zs_path, index=False)
print(f"\nV2 ZS 저장: {zs_path}")
print(f"  mean={zs_scores.mean():.4f}, fake%={(zs_scores>0.5).mean()*100:.1f}%")
print(f"  range=[{zs_scores.min():.4f}, {zs_scores.max():.4f}]")

del zs_model, zs_preprocess, zs_tokenizer
gc.collect(); torch.cuda.empty_cache()
print("V2 ZS 모델 메모리 해제 완료")

---

## Part B: 학습 데이터 다운로드 + Face Detection 준비

V3와 V5가 사용하는 **5개 데이터 소스**를 모두 다운로드합니다.

| 소스 | Kaggle Dataset | 사용 |
|------|---------------|------|
| StyleGAN | `xhlulu/140k-real-and-fake-faces` | V3, V5 |
| FF++/CelebDF | `jimmy98/faceforensics-celebv1-and-v2-df-combined-dataset` | V3 only |
| DFDC | `manjilkarki/deepfake-and-real-images` | V3, V5 |
| ciplab | `ciplab/real-and-fake-face-detection` | V5 only |
| FFHQ | `xhlulu/flickrfaceshq-dataset-nvidia-resized-256px` | V5 only |

In [ ]:
# ============================================================
# 5. 학습 데이터 다운로드 (5개 소스)
# ============================================================
import kagglehub

print("=" * 80)
print("Part B: 학습 데이터 다운로드")
print("=" * 80)

# DS1: StyleGAN (V3 + V5 공통)
print("\n[1/5] StyleGAN...")
ds_stylegan = Path(kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces"))
print(f"  → {ds_stylegan}")

# DS2: FF++/CelebDF (V3 only)
print("\n[2/5] FF++/CelebDF...")
try:
    ds_ffcdf = Path(kagglehub.dataset_download("jimmy98/faceforensics-celebv1-and-v2-df-combined-dataset"))
    print(f"  → {ds_ffcdf}")
except Exception as e:
    ds_ffcdf = None
    print(f"  ⚠ 다운로드 실패 (V3에서 제외됨): {e}")

# DS3: DFDC (V3 + V5 공통)
print("\n[3/5] DFDC...")
ds_dfdc = Path(kagglehub.dataset_download("manjilkarki/deepfake-and-real-images"))
print(f"  → {ds_dfdc}")

# DS4: ciplab (V5 only)
print("\n[4/5] ciplab...")
ds_ciplab = Path(kagglehub.dataset_download("ciplab/real-and-fake-face-detection"))
print(f"  → {ds_ciplab}")

# DS5: FFHQ (V5 only)
print("\n[5/5] FFHQ...")
ds_ffhq = Path(kagglehub.dataset_download("xhlulu/flickrfaceshq-dataset-nvidia-resized-256px"))
print(f"  → {ds_ffhq}")

print("\n모든 데이터 다운로드 완료!")

In [ ]:
# ============================================================
# 6. Face Detection 모델 준비 (MediaPipe Tasks API)
# ============================================================

FACE_MODEL_PATH = str(CACHE_DIR / "blaze_face_short_range.tflite")
if not Path(FACE_MODEL_PATH).exists():
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, FACE_MODEL_PATH)
    print(f"모델 다운로드 완료: {FACE_MODEL_PATH}")

def create_face_detector():
    base_options = mp.tasks.BaseOptions(model_asset_path=FACE_MODEL_PATH)
    options = mp_vision.FaceDetectorOptions(
        base_options=base_options,
        min_detection_confidence=0.3,
    )
    return mp_vision.FaceDetector.create_from_options(options)

_detector = None

def get_detector():
    global _detector
    if _detector is None:
        _detector = create_face_detector()
    return _detector

def close_detector():
    global _detector
    if _detector is not None:
        _detector.close()
        _detector = None

def detect_faces(rgb_np):
    det = get_detector()
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb_np))
    return det.detect(mp_image).detections

def has_face(img_path, min_conf=0.5):
    img_cv = cv2.imread(str(img_path))
    if img_cv is None:
        return False
    h, w = img_cv.shape[:2]
    if max(h, w) > 512:
        s = 512 / max(h, w)
        img_cv = cv2.resize(img_cv, (int(w * s), int(h * s)))
    rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    dets = detect_faces(rgb)
    return bool(dets and dets[0].categories[0].score >= min_conf)

def face_crop_pil(img_path, target_size=336, margin=0.3):
    img_cv = cv2.imread(str(img_path))
    if img_cv is None:
        return None
    h, w = img_cv.shape[:2]
    scale = 1.0
    if max(h, w) > 1024:
        scale = 1024 / max(h, w)
        det_img = cv2.resize(img_cv, (int(w * scale), int(h * scale)))
    else:
        det_img = img_cv

    rgb_det = cv2.cvtColor(det_img, cv2.COLOR_BGR2RGB)
    rgb_full = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    dets = detect_faces(rgb_det)

    if not dets:
        img_pil = Image.fromarray(rgb_full)
        min_dim = min(w, h)
        left = (w - min_dim) // 2; top = (h - min_dim) // 2
        img_pil = img_pil.crop((left, top, left + min_dim, top + min_dim))
        return img_pil.resize((target_size, target_size), Image.LANCZOS)

    bbox = dets[0].bounding_box
    x, y, bw, bh = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height
    if scale != 1.0:
        x = int(x / scale); y = int(y / scale)
        bw = int(bw / scale); bh = int(bh / scale)
    mx, my = int(bw * margin), int(bh * margin)

    x1 = max(0, x - mx); y1 = max(0, y - my)
    x2 = min(w, x + bw + mx); y2 = min(h, y + bh + my)

    cw, ch = x2 - x1, y2 - y1
    if cw > ch:
        d = cw - ch; y1 = max(0, y1 - d // 2); y2 = min(h, y2 + d // 2)
    elif ch > cw:
        d = ch - cw; x1 = max(0, x1 - d // 2); x2 = min(w, x2 + d // 2)

    cropped = rgb_full[y1:y2, x1:x2]
    return Image.fromarray(cropped).resize((target_size, target_size), Image.LANCZOS)

print("Face detector 준비 완료")

---

## Part C: V3 Face-Centric Fine-Tuning → `V3_face_centric_tta.csv`

CLIP ViT-L-14 (224px)를 face-centric 접근으로 fine-tune합니다.

**V5와의 차이점**:
- 224px 입력 (V5: 336px)
- FF++/CelebDF 데이터 추가 사용
- 테스트셋 얼굴 분포와 매칭되는 학습 데이터 선별 (Distribution Matching)
- 간단한 augmentation (V5: deepfake-specific augmentation)

In [ ]:
# ============================================================
# 7. V3 데이터 수집 + Face Filter + Distribution Matching
# ============================================================
print("=" * 80)
print("Part C: V3 Face-Centric Pipeline")
print("=" * 80)

random.seed(SEED)

# --- 테스트셋 얼굴 분포 분석 ---
print("\n테스트셋 얼굴 분석 중...")
test_face_areas = []
for f in tqdm(test_files, desc="Test face analysis"):
    img_cv = cv2.imread(str(f))
    if img_cv is None:
        continue
    h, w = img_cv.shape[:2]
    if max(h, w) > 512:
        s = 512 / max(h, w)
        img_cv = cv2.resize(img_cv, (int(w * s), int(h * s)))
    rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    dets = detect_faces(rgb)
    if dets and dets[0].categories[0].score >= 0.3:
        bbox = dets[0].bounding_box
        img_h, img_w = rgb.shape[:2]
        rel_area = (bbox.width * bbox.height) / (img_w * img_h)
        test_face_areas.append(rel_area)

test_mean_area = np.mean(test_face_areas)
test_std_area = np.std(test_face_areas)
print(f"테스트셋 얼굴: {len(test_face_areas)}/{len(test_files)}")
print(f"  face_area: mean={test_mean_area:.4f}, std={test_std_area:.4f}")

# --- V3 학습 데이터 수집 (3 소스: StyleGAN, FF++/CelebDF, DFDC) ---
print("\nV3 학습 데이터 수집 중...")
v3_entries = []

# StyleGAN
ds1_base = ds_stylegan / "real_vs_fake" / "real-vs-fake"
for split in ["train", "valid"]:
    for lbl in ["real", "fake"]:
        d = ds1_base / split / lbl
        if d.exists():
            files = sorted(d.glob("*.jpg")); random.shuffle(files)
            for f in files[:3000]:
                v3_entries.append((str(f), 0 if lbl == "real" else 1, "stylegan"))

# FF++/CelebDF
if ds_ffcdf is not None:
    for sub in ds_ffcdf.iterdir():
        if sub.is_dir():
            lbl = 0 if sub.name.lower() == "real" else 1
            files = sorted([f for f in sub.iterdir() if f.suffix.lower() in (".png", ".jpg", ".jpeg")])
            random.shuffle(files)
            for f in files[:3000]:
                v3_entries.append((str(f), lbl, "ff_cdf"))

# DFDC
ds3_base = ds_dfdc / "Dataset"
for d in ds3_base.iterdir():
    if d.is_dir() and d.name.lower() in ("train", "validation"):
        for sub in d.iterdir():
            if sub.is_dir():
                lbl = 0 if sub.name.lower() == "real" else 1
                files = sorted([f for f in sub.iterdir() if f.suffix.lower() in (".png", ".jpg", ".jpeg")])
                random.shuffle(files)
                for f in files[:3000]:
                    v3_entries.append((str(f), lbl, "dfdc"))

print(f"V3 수집: {len(v3_entries)}")
for src, cnt in Counter(s for _, _, s in v3_entries).items():
    r = sum(1 for _, l, ss in v3_entries if ss == src and l == 0)
    f_ = sum(1 for _, l, ss in v3_entries if ss == src and l == 1)
    print(f"  {src}: {cnt} (real={r}, fake={f_})")

# --- Face filter ---
print("\nFace filtering...")
v3_filtered = []
for path, label, source in tqdm(v3_entries, desc="V3 face filter"):
    try:
        img_cv = cv2.imread(path)
        if img_cv is None:
            continue
        h, w = img_cv.shape[:2]
        if max(h, w) > 512:
            s = 512 / max(h, w)
            img_cv = cv2.resize(img_cv, (int(w * s), int(h * s)))
        rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        dets = detect_faces(rgb)
        if not dets:
            continue
        conf = dets[0].categories[0].score
        if conf < 0.5:
            continue
        bbox = dets[0].bounding_box
        img_h, img_w = rgb.shape[:2]
        face_area = (bbox.width * bbox.height) / (img_w * img_h)
        v3_filtered.append({"path": path, "label": label, "source": source,
                            "face_conf": conf, "face_area": face_area})
    except Exception:
        continue

print(f"Face filter 통과: {len(v3_filtered)}/{len(v3_entries)}")

# --- Distribution matching ---
print("\nDistribution matching...")
scored = []
for d in v3_filtered:
    area_diff = abs(d["face_area"] - test_mean_area) / (test_std_area + 0.01)
    score = 1.0 / (1.0 + area_diff) * d["face_conf"]
    scored.append((d, score))
scored.sort(key=lambda x: x[1], reverse=True)

MAX_V3 = 8000
v3_real, v3_fake = [], []
for d, score in scored:
    if d["label"] == 0 and len(v3_real) < MAX_V3 // 2:
        v3_real.append(d)
    elif d["label"] == 1 and len(v3_fake) < MAX_V3 // 2:
        v3_fake.append(d)
    if len(v3_real) >= MAX_V3 // 2 and len(v3_fake) >= MAX_V3 // 2:
        break

v3_matched = v3_real + v3_fake
random.shuffle(v3_matched)

v3_val_size = int(len(v3_matched) * 0.1)
v3_val_data = v3_matched[:v3_val_size]
v3_train_data = v3_matched[v3_val_size:]
print(f"V3 matched: {len(v3_matched)} (real={len(v3_real)}, fake={len(v3_fake)})")
print(f"  Train: {len(v3_train_data)}, Val: {len(v3_val_data)}")

In [ ]:
# ============================================================
# 8. V3 모델 학습 (CLIP ViT-L-14, 224px)
# ============================================================
print("\n" + "=" * 80)
print("V3 Model Training")
print("=" * 80)

# --- Dataset ---
class V3Dataset(Dataset):
    def __init__(self, data_list, transform, target_size=224):
        self.data = data_list
        self.transform = transform
        self.target_size = target_size
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        try:
            img = Image.open(item["path"]).convert("RGB")
            img = img.resize((self.target_size, self.target_size), Image.LANCZOS)
        except Exception:
            img = Image.new("RGB", (self.target_size, self.target_size))
        img = self.transform(img)
        return img, torch.tensor(item["label"], dtype=torch.float32)

v3_train_tfm = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomResizedCrop(V3_IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

v3_val_tfm = transforms.Compose([
    transforms.Resize(V3_IMG_SIZE),
    transforms.CenterCrop(V3_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

v3_train_ds = V3Dataset(v3_train_data, v3_train_tfm, V3_IMG_SIZE)
v3_val_ds   = V3Dataset(v3_val_data, v3_val_tfm, V3_IMG_SIZE)

v3_train_loader = DataLoader(v3_train_ds, batch_size=V3_BATCH_SIZE, shuffle=True,
                             num_workers=4, pin_memory=True)
v3_val_loader   = DataLoader(v3_val_ds, batch_size=V3_BATCH_SIZE, shuffle=False,
                             num_workers=4, pin_memory=True)

print(f"V3 Train: {len(v3_train_ds)}, Val: {len(v3_val_ds)}")

# --- Model ---
class CLIPDetector(nn.Module):
    def __init__(self, clip_model, feat_dim=768):
        super().__init__()
        self.backbone = clip_model.visual
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        with torch.amp.autocast("cuda"):
            features = self.backbone(x)
            features = features.float()
        return self.head(features).squeeze(-1)

clip_v3, _, _ = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
v3_model = CLIPDetector(clip_v3)
v3_model = v3_model.to(DEVICE)
del clip_v3; gc.collect(); torch.cuda.empty_cache()

print(f"V3 파라미터: {sum(p.numel() for p in v3_model.parameters()) / 1e6:.1f}M")

# --- Optimizer ---
v3_optimizer = torch.optim.AdamW([
    {"params": list(v3_model.backbone.parameters()), "lr": V3_BACKBONE_LR, "weight_decay": V3_WEIGHT_DECAY},
    {"params": list(v3_model.head.parameters()),     "lr": V3_HEAD_LR,     "weight_decay": V3_WEIGHT_DECAY},
])

v3_total_steps  = len(v3_train_loader) * V3_EPOCHS
v3_warmup_steps = len(v3_train_loader) * V3_WARMUP_EPOCHS

def v3_lr_lambda(step):
    if step < v3_warmup_steps:
        return step / max(v3_warmup_steps, 1)
    progress = (step - v3_warmup_steps) / max(v3_total_steps - v3_warmup_steps, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))

v3_scheduler = torch.optim.lr_scheduler.LambdaLR(v3_optimizer, v3_lr_lambda)
v3_criterion = nn.BCEWithLogitsLoss()
v3_scaler    = GradScaler("cuda")

# --- Training loop ---
v3_best_auc = 0
v3_best_epoch = 0

for epoch in range(V3_EPOCHS):
    v3_model.train()
    train_loss = 0; train_correct = 0; train_total = 0

    pbar = tqdm(v3_train_loader, desc=f"V3 Epoch {epoch+1}/{V3_EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        v3_optimizer.zero_grad()
        with autocast("cuda"):
            logits = v3_model(imgs)
            loss = v3_criterion(logits, labels)
        v3_scaler.scale(loss).backward()
        v3_scaler.unscale_(v3_optimizer)
        torch.nn.utils.clip_grad_norm_(v3_model.parameters(), 1.0)
        v3_scaler.step(v3_optimizer); v3_scaler.update(); v3_scheduler.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += ((torch.sigmoid(logits) > 0.5).float() == labels).sum().item()
        train_total += imgs.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{train_correct/train_total:.4f}")

    # Validation
    v3_model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for imgs, labels in v3_val_loader:
            imgs = imgs.to(DEVICE)
            probs = torch.sigmoid(v3_model(imgs)).cpu().numpy()
            vp.extend(probs); vl.extend(labels.numpy())

    vp_arr, vl_arr = np.array(vp), np.array(vl)
    val_auc = roc_auc_score(vl_arr, vp_arr)
    val_acc = ((vp_arr > 0.5) == vl_arr).mean()

    print(f"V3 Epoch {epoch+1}: loss={train_loss/train_total:.4f}, acc={train_correct/train_total:.4f}")
    print(f"  val_auc={val_auc:.4f}, val_acc={val_acc:.4f}")

    if val_auc > v3_best_auc:
        v3_best_auc = val_auc
        v3_best_epoch = epoch + 1
        torch.save(v3_model.state_dict(), CKPT_DIR / "v3_face_centric_best.pt")
        print(f"  ★ New best!")

print(f"\nV3 Best val AUC: {v3_best_auc:.4f} at epoch {v3_best_epoch}")

In [ ]:
# ============================================================
# 9. V3 테스트 추론 (TTA) → V3_face_centric_tta.csv
# ============================================================
print("\n" + "=" * 80)
print("V3 Test Inference with TTA")
print("=" * 80)

v3_model.load_state_dict(torch.load(CKPT_DIR / "v3_face_centric_best.pt", weights_only=True))
v3_model.eval()

# Face crop for test images (224px)
print("V3 테스트 이미지 face crop (224px)...")
v3_test_crops = []
for f in tqdm(test_files, desc="V3 test face crop"):
    try:
        img = face_crop_pil(f, target_size=V3_IMG_SIZE)
        if img is None:
            img = Image.open(f).convert("RGB").resize((V3_IMG_SIZE, V3_IMG_SIZE), Image.LANCZOS)
        v3_test_crops.append(img)
    except Exception:
        v3_test_crops.append(Image.new("RGB", (V3_IMG_SIZE, V3_IMG_SIZE)))

# TTA transforms
v3_tta_tfms = [
    v3_val_tfm,
    transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.Resize(V3_IMG_SIZE),
        transforms.CenterCrop(V3_IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize(int(V3_IMG_SIZE * 1.1)),
        transforms.CenterCrop(V3_IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize(int(V3_IMG_SIZE * 0.9)),
        transforms.CenterCrop(int(V3_IMG_SIZE * 0.9)),
        transforms.Resize(V3_IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
]

v3_all_tta = []
for t_idx, tfm in enumerate(v3_tta_tfms):
    preds = []
    with torch.no_grad():
        for i in range(0, len(v3_test_crops), 16):
            batch_imgs = v3_test_crops[i:i+16]
            tensors = torch.stack([tfm(img.copy()) for img in batch_imgs]).to(DEVICE)
            probs = torch.sigmoid(v3_model(tensors)).cpu().numpy()
            preds.extend(probs)
    preds = np.array(preds)
    v3_all_tta.append(preds)
    print(f"  V3 TTA {t_idx+1}: mean={preds.mean():.4f}, fake%={(preds>0.5).mean()*100:.1f}%")

v3_tta_preds = np.mean(v3_all_tta, axis=0)

# 저장
v3_df = pd.DataFrame({"id": test_ids, "score": v3_tta_preds})
v3_path = SUB_DIR / "V3_face_centric_tta.csv"
v3_df.to_csv(v3_path, index=False)
print(f"\nV3 저장: {v3_path}")
print(f"  mean={v3_tta_preds.mean():.4f}, fake%={(v3_tta_preds>0.5).mean()*100:.1f}%")

# 메모리 정리
del v3_model, v3_train_ds, v3_val_ds, v3_train_loader, v3_val_loader
del v3_test_crops, v3_all_tta, v3_tta_tfms
del v3_entries, v3_filtered, v3_matched, v3_train_data, v3_val_data
gc.collect(); torch.cuda.empty_cache()
close_detector()
print("V3 메모리 해제 완료")

---

## Part D: V5 Main Pipeline → `V5_final_tta.csv`

CLIP ViT-L-14-**336** (336px 입력)을 deepfake-specific augmentation으로 fine-tune합니다.

| 항목 | V3 | V5 |
|------|----|----|
| 입력 크기 | 224px | 336px |
| 백본 | ViT-L-14 | ViT-L-14-336 |
| 데이터 | StyleGAN+FF++/CelebDF+DFDC | StyleGAN+DFDC+ciplab+FFHQ |
| Augmentation | 기본 | JPEG 압축, Downscale-Upscale 등 |
| Epochs | 8 | 12 |

In [ ]:
# ============================================================
# 10. V5 데이터 수집 + Face Filter + Pre-crop (336px)
# ============================================================
print("=" * 80)
print("Part D: V5 Main Pipeline (336px)")
print("=" * 80)

random.seed(SEED)
v5_entries = []

# StyleGAN
ds1_base = ds_stylegan / "real_vs_fake" / "real-vs-fake"
for split in ["train", "valid"]:
    for lbl in ["real", "fake"]:
        d = ds1_base / split / lbl
        if d.exists():
            files = sorted(d.glob("*.jpg")); random.shuffle(files)
            for f in files[:3000]:
                v5_entries.append((str(f), 0 if lbl == "real" else 1, "stylegan"))

# DFDC
ds3_base = ds_dfdc / "Dataset"
for d in ds3_base.iterdir():
    if d.is_dir() and d.name.lower() in ("train", "validation"):
        for sub in d.iterdir():
            if sub.is_dir():
                lbl = 0 if sub.name.lower() == "real" else 1
                files = sorted([f for f in sub.iterdir() if f.suffix.lower() in (".png", ".jpg")])
                random.shuffle(files)
                for f in files[:3000]:
                    v5_entries.append((str(f), lbl, "dfdc"))

# ciplab
ds_ciplab_face = ds_ciplab / "real_and_fake_face"
if ds_ciplab_face.exists():
    for sub in ds_ciplab_face.iterdir():
        if sub.is_dir():
            lbl = 0 if "real" in sub.name.lower() else 1
            files = sorted(sub.rglob("*.jpg")) + sorted(sub.rglob("*.png"))
            for f in files:
                v5_entries.append((str(f), lbl, "ciplab"))

# FFHQ (real only)
ffhq_files = list(ds_ffhq.rglob("*.png")) + list(ds_ffhq.rglob("*.jpg"))
random.shuffle(ffhq_files)
for f in ffhq_files[:4000]:
    v5_entries.append((str(f), 0, "ffhq"))

print(f"V5 수집: {len(v5_entries)}")
for src, cnt in Counter(s for _, _, s in v5_entries).items():
    r = sum(1 for _, l, ss in v5_entries if ss == src and l == 0)
    f_ = sum(1 for _, l, ss in v5_entries if ss == src and l == 1)
    print(f"  {src}: {cnt} (real={r}, fake={f_})")

# Face filter + pre-crop (336px)
print("\nFace filter + pre-crop (336px)...")
v5_cropped = []
no_face = 0

for path, label, source in tqdm(v5_entries, desc="V5 face filter+crop"):
    try:
        if not has_face(path):
            no_face += 1
            continue
        img = face_crop_pil(path, target_size=V5_IMG_SIZE)
        if img is None:
            continue
        v5_cropped.append((img, label, source))
    except Exception:
        continue

close_detector()
print(f"\nV5 face filter: {len(v5_cropped)}/{len(v5_entries)} (no_face={no_face})")
for src in set(s for _, _, s in v5_cropped):
    r = sum(1 for _, l, ss in v5_cropped if ss == src and l == 0)
    f_ = sum(1 for _, l, ss in v5_cropped if ss == src and l == 1)
    print(f"  {src}: real={r}, fake={f_}")

In [ ]:
# ============================================================
# 11. V5 클래스 밸런싱 + Augmentation + DataLoader
# ============================================================
random.seed(SEED)

reals = [(img, l, s) for img, l, s in v5_cropped if l == 0]
fakes = [(img, l, s) for img, l, s in v5_cropped if l == 1]
random.shuffle(reals); random.shuffle(fakes)

n_per_class = min(len(reals), len(fakes), 6000)
reals = reals[:n_per_class]; fakes = fakes[:n_per_class]
v5_all_data = reals + fakes
random.shuffle(v5_all_data)

v5_val_size = int(len(v5_all_data) * 0.1)
v5_val_data = v5_all_data[:v5_val_size]
v5_train_data = v5_all_data[v5_val_size:]

print(f"클래스당: {n_per_class} | Train: {len(v5_train_data)} | Val: {len(v5_val_data)}")

# Deepfake-specific augmentation
class JPEGCompression:
    def __init__(self, quality_range=(30, 95)):
        self.lo, self.hi = quality_range
    def __call__(self, img):
        q = random.randint(self.lo, self.hi)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

class DownscaleUpscale:
    def __init__(self, scale_range=(0.4, 0.8)):
        self.lo, self.hi = scale_range
    def __call__(self, img):
        w, h = img.size
        s = random.uniform(self.lo, self.hi)
        small = img.resize((int(w * s), int(h * s)), Image.BILINEAR)
        return small.resize((w, h), Image.BILINEAR)

class GaussianNoise:
    def __init__(self, std_range=(0.01, 0.05)):
        self.lo, self.hi = std_range
    def __call__(self, tensor):
        std = random.uniform(self.lo, self.hi)
        return tensor + torch.randn_like(tensor) * std

v5_train_tfm = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomResizedCrop(V5_IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.RandomApply([JPEGCompression(quality_range=(30, 95))], p=0.3),
    transforms.RandomApply([DownscaleUpscale(scale_range=(0.4, 0.8))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.RandomApply([GaussianNoise(std_range=(0.01, 0.03))], p=0.15),
])

v5_val_tfm = transforms.Compose([
    transforms.Resize((V5_IMG_SIZE, V5_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

class PrecroppedDataset(Dataset):
    def __init__(self, data_list, transform):
        self.data = data_list
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        img, label, source = self.data[idx]
        try:
            img = self.transform(img.copy())
        except Exception:
            img = torch.zeros(3, V5_IMG_SIZE, V5_IMG_SIZE)
        return img, torch.tensor(label, dtype=torch.float32)

v5_train_ds = PrecroppedDataset(v5_train_data, v5_train_tfm)
v5_val_ds   = PrecroppedDataset(v5_val_data, v5_val_tfm)

v5_train_loader = DataLoader(v5_train_ds, batch_size=V5_BATCH_SIZE, shuffle=True,
                             num_workers=4, pin_memory=True, persistent_workers=True)
v5_val_loader   = DataLoader(v5_val_ds, batch_size=V5_BATCH_SIZE, shuffle=False,
                             num_workers=4, pin_memory=True, persistent_workers=True)

print(f"V5 Train: {len(v5_train_ds)} samples, {len(v5_train_loader)} batches")
print(f"V5 Val: {len(v5_val_ds)} samples, {len(v5_val_loader)} batches")

In [ ]:
# ============================================================
# 12. V5 모델 학습 (CLIP ViT-L-14-336, 336px)
# ============================================================
print("=" * 80)
print("V5 Model Training")
print("=" * 80)

clip_v5, _, _ = open_clip.create_model_and_transforms("ViT-L-14-336", pretrained="openai")
v5_model = CLIPDetector(clip_v5)
v5_model = v5_model.to(DEVICE)
del clip_v5; gc.collect(); torch.cuda.empty_cache()

print(f"V5 파라미터: {sum(p.numel() for p in v5_model.parameters()) / 1e6:.1f}M")

v5_optimizer = torch.optim.AdamW([
    {"params": list(v5_model.backbone.parameters()), "lr": V5_BACKBONE_LR, "weight_decay": V5_WEIGHT_DECAY},
    {"params": list(v5_model.head.parameters()),     "lr": V5_HEAD_LR,     "weight_decay": V5_WEIGHT_DECAY},
])

v5_total_steps  = len(v5_train_loader) * V5_EPOCHS
v5_warmup_steps = len(v5_train_loader) * 2

def v5_lr_lambda(step):
    if step < v5_warmup_steps:
        return step / max(v5_warmup_steps, 1)
    progress = (step - v5_warmup_steps) / max(v5_total_steps - v5_warmup_steps, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))

v5_scheduler = torch.optim.lr_scheduler.LambdaLR(v5_optimizer, v5_lr_lambda)
v5_criterion = nn.BCEWithLogitsLoss()
v5_scaler    = GradScaler("cuda")

best_val_auc = 0
no_improve = 0

for epoch in range(V5_EPOCHS):
    v5_model.train()
    train_loss = 0; train_correct = 0; train_total = 0

    pbar = tqdm(v5_train_loader, desc=f"V5 Epoch {epoch+1}/{V5_EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        v5_optimizer.zero_grad()
        with autocast("cuda"):
            logits = v5_model(imgs)
            loss = v5_criterion(logits, labels)
        v5_scaler.scale(loss).backward()
        v5_scaler.unscale_(v5_optimizer)
        torch.nn.utils.clip_grad_norm_(v5_model.parameters(), 1.0)
        v5_scaler.step(v5_optimizer); v5_scaler.update(); v5_scheduler.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += ((torch.sigmoid(logits) > 0.5).float() == labels).sum().item()
        train_total += imgs.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{train_correct/train_total:.4f}")

    # Validation
    v5_model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for imgs, labels in v5_val_loader:
            imgs = imgs.to(DEVICE)
            probs = torch.sigmoid(v5_model(imgs)).cpu().numpy()
            vp.extend(probs); vl.extend(labels.numpy())

    vp_arr, vl_arr = np.array(vp), np.array(vl)
    val_auc = roc_auc_score(vl_arr, vp_arr)
    val_acc = ((vp_arr > 0.5) == vl_arr).mean()

    print(f"V5 Epoch {epoch+1}: loss={train_loss/train_total:.4f}, acc={train_correct/train_total:.4f}")
    print(f"  val_auc={val_auc:.4f}, val_acc={val_acc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(v5_model.state_dict(), CKPT_DIR / "v5_final_best.pt")
        print(f"  ★ New best!")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= V5_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

print(f"\nV5 Best val AUC: {best_val_auc:.4f}")
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 13. V5 테스트 추론 (Face Crop + 5-view TTA) → V5_final_tta.csv
# ============================================================
print("=" * 80)
print("V5 Test Inference with TTA")
print("=" * 80)

v5_model.load_state_dict(torch.load(CKPT_DIR / "v5_final_best.pt", weights_only=True))
v5_model.eval()

# 테스트 이미지 face crop (336px)
print("V5 테스트 이미지 face crop (336px)...")
v5_test_crops = []

for f in tqdm(test_files, desc="V5 test face crop"):
    try:
        img = face_crop_pil(f, target_size=V5_IMG_SIZE)
        if img is None:
            img = Image.open(f).convert("RGB")
            w, h = img.size
            min_dim = min(w, h)
            left = (w - min_dim) // 2; top = (h - min_dim) // 2
            img = img.crop((left, top, left + min_dim, top + min_dim))
            img = img.resize((V5_IMG_SIZE, V5_IMG_SIZE), Image.LANCZOS)
        v5_test_crops.append(img)
    except Exception:
        v5_test_crops.append(Image.new("RGB", (V5_IMG_SIZE, V5_IMG_SIZE)))

close_detector()

# Normal inference
print("\nNormal inference...")
normal_preds = []
with torch.no_grad():
    for i in range(0, len(v5_test_crops), 16):
        batch_imgs = v5_test_crops[i:i+16]
        tensors = torch.stack([v5_val_tfm(img) for img in batch_imgs]).to(DEVICE)
        probs = torch.sigmoid(v5_model(tensors)).cpu().numpy()
        normal_preds.extend(probs)
normal_preds = np.array(normal_preds)

# TTA: 5 views
print("TTA inference (5 views)...")
v5_tta_tfms = [
    v5_val_tfm,
    transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.Resize((V5_IMG_SIZE, V5_IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize((int(V5_IMG_SIZE * 1.1), int(V5_IMG_SIZE * 1.1))),
        transforms.CenterCrop(V5_IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize((int(V5_IMG_SIZE * 0.9), int(V5_IMG_SIZE * 0.9))),
        transforms.Resize((V5_IMG_SIZE, V5_IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
    transforms.Compose([
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.Resize((V5_IMG_SIZE, V5_IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]),
]

v5_all_tta = []
for t_idx, tfm in enumerate(v5_tta_tfms):
    preds = []
    with torch.no_grad():
        for i in range(0, len(v5_test_crops), 16):
            batch_imgs = v5_test_crops[i:i+16]
            tensors = torch.stack([tfm(img.copy()) for img in batch_imgs]).to(DEVICE)
            probs = torch.sigmoid(v5_model(tensors)).cpu().numpy()
            preds.extend(probs)
    preds = np.array(preds)
    v5_all_tta.append(preds)
    print(f"  V5 TTA {t_idx+1}: mean={preds.mean():.4f}, fake%={(preds>0.5).mean()*100:.1f}%")

v5_tta_preds = np.mean(v5_all_tta, axis=0)

# 저장
v5_df = pd.DataFrame({"id": test_ids, "score": v5_tta_preds})
v5_df = sample_sub[["id"]].merge(v5_df, on="id", how="left")
v5_path = SUB_DIR / "V5_final_tta.csv"
v5_df.to_csv(v5_path, index=False)
print(f"\nV5 저장: {v5_path}")
print(f"  mean={v5_tta_preds.mean():.4f}, fake%={(v5_tta_preds>0.5).mean()*100:.1f}%")

# 메모리 정리
del v5_model, v5_test_crops, v5_all_tta
gc.collect(); torch.cuda.empty_cache()
print("V5 메모리 해제 완료")

---

## Part E: 3-way Rank Ensemble → `V5_ENS_zs_dom_3way.csv`

세 모델의 예측을 Rank Normalization 후 가중 평균합니다:

```
V5_ENS_zs_dom_3way = 0.50 × rank(ZS) + 0.25 × rank(V5) + 0.25 × rank(V3)
```

- **Rank Normalization**: 값 자체가 아닌 순위를 [0,1]로 정규화 → 스케일 불변
- **ZS에 50% 가중치**: 학습 데이터 편향이 없는 범용 특징
- **V5 + V3에 25%씩**: fine-tuned 모델이 포착하는 미세한 패턴

In [ ]:
# ============================================================
# 14. 3-way Rank Ensemble → V5_ENS_zs_dom_3way.csv
# ============================================================
print("=" * 80)
print("Part E: 3-way Ensemble")
print("=" * 80)

def rank_norm(s):
    s = np.array(s, dtype=float)
    return s.argsort().argsort().astype(float) / (len(s) - 1)

def save_sub(scores, name):
    df = pd.DataFrame({"id": test_ids, "score": scores})
    df = sample_sub[["id"]].merge(df, on="id", how="left")
    path = SUB_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    s = np.array(scores)
    print(f"  [{name}] mean={s.mean():.4f}, fake%={(s>0.5).mean()*100:.1f}%, range=[{s.min():.4f}, {s.max():.4f}]")
    return path

# 이 노트북에서 생성한 세 CSV 로드
zs_path = SUB_DIR / "sub_V2_clip_L14_v3.csv"
v3_path = SUB_DIR / "V3_face_centric_tta.csv"
v5_path = SUB_DIR / "V5_final_tta.csv"

assert zs_path.exists(), f"V2 ZS CSV가 없습니다: {zs_path}"
assert v3_path.exists(), f"V3 CSV가 없습니다: {v3_path}"
assert v5_path.exists(), f"V5 CSV가 없습니다: {v5_path}"

zs_scores_loaded = pd.read_csv(zs_path)["score"].values
v3_scores_loaded = pd.read_csv(v3_path)["score"].values
v5_scores_loaded = pd.read_csv(v5_path)["score"].values

print(f"ZS: mean={zs_scores_loaded.mean():.4f}, fake%={(zs_scores_loaded>0.5).mean()*100:.1f}%")
print(f"V3: mean={v3_scores_loaded.mean():.4f}, fake%={(v3_scores_loaded>0.5).mean()*100:.1f}%")
print(f"V5: mean={v5_scores_loaded.mean():.4f}, fake%={(v5_scores_loaded>0.5).mean()*100:.1f}%")

# Rank normalize
zsr = rank_norm(zs_scores_loaded)
v3r = rank_norm(v3_scores_loaded)
v5r = rank_norm(v5_scores_loaded)

# V5 단독
save_sub(v5_scores_loaded, "V5_final_tta_standalone")

# V5 + ZS
save_sub(0.6 * v5r + 0.4 * zsr, "V5_ENS_v5_60_zs_40")
save_sub(0.6 * zsr + 0.4 * v5r, "V5_ENS_zs_60_v5_40")

# 3-way
save_sub(0.4 * v5r + 0.3 * v3r + 0.3 * zsr, "V5_ENS_3way")

# ★ 목표: 0.50 ZS + 0.25 V5 + 0.25 V3
ens_final = 0.50 * zsr + 0.25 * v5r + 0.25 * v3r
save_sub(ens_final, "V5_ENS_zs_dom_3way")

print("\n모든 앙상블 파일 생성 완료!")

In [ ]:
# ============================================================
# 15. 최종 검증
# ============================================================
print("=" * 80)
print("최종 검증")
print("=" * 80)

target = SUB_DIR / "V5_ENS_zs_dom_3way.csv"
if target.exists():
    df = pd.read_csv(target)
    print(f"파일: {target}")
    print(f"행 수: {len(df)}")
    print(f"Score 범위: [{df['score'].min():.6f}, {df['score'].max():.6f}]")
    print(f"Score 평균: {df['score'].mean():.6f}")
    print(f"Fake 비율 (>0.5): {(df['score'] > 0.5).mean() * 100:.1f}%")
    print(f"\n상위 5행:")
    print(df.head())
    print(f"\n하위 5행:")
    print(df.tail())
else:
    print("[ERROR] V5_ENS_zs_dom_3way.csv 생성 실패!")

# 생성된 모든 파일 확인
print(f"\n{'='*40}")
print("생성된 제출 파일:")
for name in ["sub_V2_clip_L14_v3.csv", "V3_face_centric_tta.csv",
             "V5_final_tta.csv", "V5_ENS_zs_dom_3way.csv"]:
    p = SUB_DIR / name
    if p.exists():
        df = pd.read_csv(p)
        print(f"  ✓ {name}: {len(df)} rows, mean={df['score'].mean():.4f}")
    else:
        print(f"  ✗ {name}: NOT FOUND")

print(f"\n파이프라인 완료!")